# fase_3 - script_cimut Migration

This notebook handles migration of database from old DB to new DB for fase 3.

**Purpose**: Benerin database lama ke database baru untuk bagian CRM, Prospek, dan Operasional.

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
import datetime
import random
import string
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## Connect ke Database

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


## Ambil Data dari DB Lama

In [3]:
# ---------------------------------------------------------
# UPDATE: AMBIL DAFTAR TABEL SECARA DINAMIS
# ---------------------------------------------------------
cursor_old.execute("SHOW TABLES")
tables_data = cursor_old.fetchall()
target_tables = [list(t.values())[0] for t in tables_data]
print(f"\n--- Ditemukan {len(target_tables)} tabel di Database Lama ---")

df_old = {}
for table in target_tables:
    try:
        query = f"SELECT * FROM `{table}`"
        df_old[table] = pd.read_sql(query, db_old)
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_old[table])}")
    except Exception as e:
        print(f"Gagal load tabel {table}: {e}")

print("\n--- Proses load selesai. Semua data tersimpan di 'df_old' ---")


--- Ditemukan 108 tabel di Database Lama ---
Berhasil load tabel: absensi | Jumlah baris: 13444
Berhasil load tabel: absensi_note | Jumlah baris: 11
Berhasil load tabel: bidang | Jumlah baris: 4
Berhasil load tabel: bidangkategori | Jumlah baris: 12
Berhasil load tabel: bidanglink | Jumlah baris: 7
Berhasil load tabel: calon | Jumlah baris: 4
Berhasil load tabel: calon_detil | Jumlah baris: 61
Berhasil load tabel: calon_pertanyaan | Jumlah baris: 229
Berhasil load tabel: calon_pertanyaan_detil | Jumlah baris: 4305
Berhasil load tabel: catatan_kelas | Jumlah baris: 12797
Berhasil load tabel: catatan_kelas_tag | Jumlah baris: 999
Berhasil load tabel: catatan_mingguan | Jumlah baris: 0
Berhasil load tabel: catatan_siswa | Jumlah baris: 1502
Berhasil load tabel: catatan_siswa_follow_up | Jumlah baris: 22
Berhasil load tabel: catatanawal_admin | Jumlah baris: 64
Berhasil load tabel: catatanawal_datautama | Jumlah baris: 9
Berhasil load tabel: catatanawal_infolain | Jumlah baris: 64
Berhasi

## Ambil Data dari DB Baru (Struktur Target)

In [4]:
cursor_new.execute("SHOW TABLES")
tables_data_new = cursor_new.fetchall()
target_tables_new = [list(t.values())[0] for t in tables_data_new]
df_new = {}

for table in target_tables_new:
    try:
        query = f"SELECT * FROM `{table}`"
        df_new[table] = pd.read_sql(query, db_new)
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_new[table])}")
    except:
        pass

Berhasil load tabel: absensi | Jumlah baris: 0
Berhasil load tabel: activity_log | Jumlah baris: 0
Berhasil load tabel: admin_sarpras | Jumlah baris: 1
Berhasil load tabel: bidang_kategori | Jumlah baris: 12
Berhasil load tabel: bidang_link | Jumlah baris: 7
Berhasil load tabel: busdev_bidang | Jumlah baris: 4
Berhasil load tabel: cache | Jumlah baris: 0
Berhasil load tabel: cache_locks | Jumlah baris: 0
Berhasil load tabel: calon_siswa | Jumlah baris: 165
Berhasil load tabel: calon_siswa_akademik | Jumlah baris: 169
Berhasil load tabel: calon_siswa_bayar | Jumlah baris: 169
Berhasil load tabel: calon_siswa_jadwal | Jumlah baris: 169
Berhasil load tabel: calon_siswa_kursus | Jumlah baris: 169
Berhasil load tabel: calon_siswa_ortu | Jumlah baris: 169
Berhasil load tabel: calon_siswa_proses | Jumlah baris: 169
Berhasil load tabel: calon_siswa_status_logs | Jumlah baris: 0
Berhasil load tabel: catatan_kelas | Jumlah baris: 0
Berhasil load tabel: catatan_kelas_tag | Jumlah baris: 0
Berhasi

# Target: izin_karyawan, verifikasi_izin, absensi, verifikasi_absensi, karyawan_resign.